# 06 · ML model end to end: the classic take-home

"Here is a customer table. Predict who will churn, and tell us what to do about it." This notebook is the whole take-home in the order a reviewer wants to read it. It starts with quick EDA, then a **leakage check**, a leakage-safe split, and a `ColumnTransformer` + `Pipeline`. Next come a baseline and three models under cross-validation, then **threshold tuning for a business cost**, feature importance, error analysis, and save/load with joblib. It ends with a `predict()` function behind a FastAPI endpoint, tested with `TestClient`.

**Time:** ~40 min · **Runs:** offline on CPU in < 1 min · **Needs:** [feature engineering and pipelines](../05_machine_learning/09_feature_engineering_and_pipelines.ipynb) and [metrics, confusion matrix, ROC/PR](../05_machine_learning/04_metrics_confusion_matrix_roc_pr.ipynb).

## Why this matters in interviews

- AI-engineer loops still include a classic ML round, as a take-home or a live notebook. It is graded on **process**, not on the last 0.5% of AUC.
- The things that separate candidates: catching **leakage**, choosing the metric and **threshold from the business cost**, using a baseline, keeping preprocessing **inside the pipeline**, and shipping something callable.
- The same discipline (held-out data, baselines, error analysis, deployment) carries straight over to LLM systems.

## What interviewers ask

This round rarely maps onto the GenAI question bank, so here are the typical asks instead:

- "Walk me through how you would approach this dataset." Start with EDA, the target, leakage, split, baseline, models, metric, threshold, errors, then ship.
- "Why this metric? Why this threshold?" Answer with costs, class imbalance, and PR versus ROC.
- "How do you know you're not leaking information?" A split before fitting, preprocessing in a `Pipeline`, and a check for features known only after the outcome.
- "Which features matter, and can I trust that?" Permutation importance on held-out data versus impurity importance.
- "How would you deploy and monitor this?" A serialised pipeline, an input schema, an endpoint, and drift and calibration checks.

## The question, as the interviewer asks it

> *"Here's an export of 3,000 subscription customers with a `churned` column. Build a model that flags customers likely to churn so the retention team can call them with an offer. The offer costs us about \$40 per customer. A churned customer costs about \$300 in lost revenue. You have two hours. Show your work."*

## The first 60 seconds

| Clarify | Why |
|---|---|
| When is each column recorded, relative to the churn event? | Anything recorded *after* churn is leakage, and it will look like your best feature. |
| What does the team do with a flag, and what does it cost? | This sets the metric and the threshold. At $300 against $40, a missed churner costs 7.5× a wasted offer. |
| How many calls can the team make per week? | Capacity turns the problem into "rank the top N", and precision@N becomes the metric. |
| Is the data a snapshot or a time series? | For time series, split by time, not randomly. |

**The plan:** "Quick EDA: the target rate, and churn by the obvious drivers. A leakage check on suspicious columns. A stratified train/test split, with the test set untouched until the end. Preprocessing in a `ColumnTransformer` inside a `Pipeline`, so cross-validation is leak-free. A dummy baseline, then logistic regression, a random forest and gradient boosting under 5-fold CV. Pick the threshold that minimises expected cost on out-of-fold predictions, then report on the test set once. Then feature importance, error analysis, and a saved pipeline behind a FastAPI endpoint."

## Packages

| Package | What it gives you | Swap it for |
|---|---|---|
| `pandas` | load, inspect and group the table | Polars |
| `scikit-learn` | `ColumnTransformer`, `Pipeline`, models, CV, metrics, `permutation_importance` | XGBoost, LightGBM, CatBoost for the models (same `fit`/`predict_proba` API) |
| `matplotlib` | EDA and diagnostic charts | seaborn, plotly |
| `joblib` | serialise the *fitted pipeline* (preprocessing + model) in one file | `skops` (safer loading), ONNX (cross-language), MLflow model registry |
| `fastapi` + `pydantic` | an input schema and the `/predict` endpoint | BentoML, a batch scoring job |

## Step 1 · The data

The table is synthetic, but shaped like the classic telco churn set: tenure, charges, contract type, payment method, internet service, support calls and a senior flag. Missing values are sprinkled into `monthly_charges`. There is also one column, `cancellation_fee`, that a careless candidate will love.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 3000
contract = rng.choice(["month-to-month", "one-year", "two-year"], n, p=[0.55, 0.25, 0.20])
payment = rng.choice(["card", "bank transfer", "electronic check", "mailed check"], n, p=[0.3, 0.25, 0.3, 0.15])
internet = rng.choice(["dsl", "fiber", "none"], n, p=[0.4, 0.45, 0.15])
tenure = np.clip(rng.exponential(24, n).round().astype(int) + 1, 1, 72)
monthly = np.round(np.clip(rng.normal(65, 25, n) + np.where(internet == "fiber", 20, 0), 18, 130), 2)
support_calls = rng.poisson(1.5, n)
senior = (rng.random(n) < 0.18).astype(int)

logit = (-2.1 + 1.4 * (contract == "month-to-month") - 0.9 * (contract == "two-year") - 0.035 * tenure
         + 0.012 * (monthly - 65) + 0.35 * support_calls + 0.45 * (internet == "fiber")
         + 0.5 * (payment == "electronic check") + 0.3 * senior)
churned = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
cancellation_fee = np.where(churned == 1, rng.choice([0, 50, 100], n, p=[0.1, 0.5, 0.4]), 0) \
                   + np.where((churned == 0) & (rng.random(n) < 0.01), 50, 0)          # recorded AFTER churn

df = pd.DataFrame({"tenure_months": tenure, "monthly_charges": monthly, "contract": contract,
                   "payment_method": payment, "internet": internet, "support_calls": support_calls,
                   "senior": senior, "cancellation_fee": cancellation_fee, "churned": churned})
df.loc[rng.random(n) < 0.03, "monthly_charges"] = np.nan
print(df.shape, "| churn rate:", f"{df.churned.mean():.1%}")
print(df.head(4).to_string())
print("\nmissing values:", df.isna().sum()[df.isna().sum() > 0].to_dict(), "| dtypes:", dict(df.dtypes.astype(str)))

## Step 2 · Quick EDA: two charts, then move on

In a take-home, EDA is for finding **target imbalance, obvious drivers, and anything suspicious**, not for a gallery of charts.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
rates = df.groupby("contract")["churned"].mean().sort_values(ascending=False)
axes[0].bar(rates.index, rates.values, color=["#e03131", "#f59f00", "#2f9e44"])
axes[0].axhline(df.churned.mean(), color="grey", ls="--", label=f"overall {df.churned.mean():.0%}")
axes[0].set_ylabel("churn rate"); axes[0].set_title("Churn by contract type"); axes[0].legend()
bins = np.arange(0, 75, 4)
for label, color in [(0, "#4c6ef5"), (1, "#e03131")]:
    axes[1].hist(df.loc[df.churned == label, "tenure_months"], bins=bins, alpha=0.6, density=True, color=color,
                 label="churned" if label else "stayed")
axes[1].set_xlabel("tenure (months)"); axes[1].set_ylabel("density"); axes[1].set_title("Churners are mostly new customers")
axes[1].legend(); plt.tight_layout(); plt.show()
print(df.groupby("churned")[["tenure_months", "monthly_charges", "support_calls", "cancellation_fee"]].mean().round(1))

Month-to-month customers churn several times more often, and churners cluster in their first year. Both are plausible drivers. Now look at the last column of that table: **`cancellation_fee` is almost always zero for customers who stayed.** A fee is charged *when someone cancels*, so the column is recorded after the outcome. It is the textbook leak. The next cell shows what happens if you keep it.

## Step 3 · The leakage check, then a leakage-safe split

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

X_all, y = df.drop(columns="churned"), df["churned"]
leaky_auc = cross_val_score(LogisticRegression(max_iter=1000), X_all[["cancellation_fee"]], y,
                            cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc").mean()
print(f"AUC from cancellation_fee ALONE: {leaky_auc:.3f}  <- too good to be true: it is recorded after churn")
assert leaky_auc > 0.9

X = X_all.drop(columns="cancellation_fee")             # drop it BEFORE anything else
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print(f"train {X_train.shape}, test {X_test.shape}; churn rate train {y_train.mean():.3f} / test {y_test.mean():.3f}")

Rules to say out loud:

- A single feature with AUC > 0.9 on a messy business problem is **a leak until proven otherwise**. Ask *when* each column is recorded.
- **Split before fitting anything.** Imputer medians, scaler means and one-hot categories must be learned from the training rows only. Putting them in a `Pipeline` guarantees that inside cross-validation too.
- `stratify=y` keeps the churn rate equal in train and test. With time-stamped data, split by date instead.

## Step 4 · `ColumnTransformer` + `Pipeline`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["tenure_months", "monthly_charges", "support_calls", "senior"]
CATEGORICAL = ["contract", "payment_method", "internet"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
])
demo = preprocess.fit(X_train)
print("features after preprocessing:", len(demo.get_feature_names_out()))
print(list(demo.get_feature_names_out()))

`handle_unknown="ignore"` matters in production. A category never seen in training (a new payment method) becomes all-zeros instead of crashing the endpoint.

## Step 5 · Baseline first, then three models under cross-validation

The **dummy baseline** predicts the base rate for everyone. Any model has to beat it clearly, or the features carry no signal. Report **ROC-AUC** (ranking quality overall) and **average precision** (PR-AUC). With a minority positive class, PR-AUC is the more honest of the two.

In [ ]:
import time

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_validate

MODELS = {
    "baseline (dummy)": DummyClassifier(strategy="prior"),
    "logistic regression": LogisticRegression(max_iter=1000, C=1.0),
    "random forest": RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=0, n_jobs=2),
    "gradient boosting": HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=0),
}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
cv_rows = []
for name, model in MODELS.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    t0 = time.perf_counter()
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["roc_auc", "average_precision"])
    cv_rows.append({"model": name, "roc_auc": res["test_roc_auc"].mean(), "roc_auc_sd": res["test_roc_auc"].std(),
                    "pr_auc": res["test_average_precision"].mean(), "seconds": time.perf_counter() - t0})
CV = pd.DataFrame(cv_rows).set_index("model")
print(CV.round(3).to_string())
assert CV.loc["baseline (dummy)", "roc_auc"] == 0.5 and CV.drop("baseline (dummy)").roc_auc.min() > 0.7

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.barh(CV.index, CV.roc_auc, xerr=CV.roc_auc_sd, color=["#adb5bd", "#4c6ef5", "#2f9e44", "#f59f00"], capsize=4)
ax.axvline(0.5, color="grey", ls=":"); ax.set_xlim(0.4, 0.9); ax.invert_yaxis()
ax.set_xlabel("ROC-AUC, 5-fold CV on the training set (bar = mean, line = ±1 sd)")
ax.set_title("Baseline vs three models: the simple one is not worse")
plt.tight_layout(); plt.show()

The three real models land within about a standard deviation of each other. That is common on tabular data with a mostly *additive* signal, which is how this data was generated. Pick **logistic regression**. It is as good, it is interpretable (its coefficients are log-odds), it is fast, and it is well calibrated, which matters because the next step uses its probabilities as costs. Say that trade-off out loud. "I picked the boosted model because it had the highest mean" is the weaker answer when the gap is inside the noise.

## Step 6 · Choose the threshold from the business cost

The model outputs probabilities, and the retention team needs a yes/no. **0.5 is not a law.** With a missed churner costing \$300 and a wasted offer \$40, the cost of a threshold `t` is:

$$\text{cost}(t) = 300 \cdot FN(t) + 40 \cdot FP(t)$$

Choose `t` on **out-of-fold** predictions from the training set, never on the test set, then report the test cost once. (If the offer only saves some of the churners, multiply \$300 by that save rate. Same method, different number.)

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score
from sklearn.model_selection import cross_val_predict

COST_FN, COST_FP = 300, 40
final = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])
oof = cross_val_predict(final, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

def total_cost(y_true, proba, t):
    tn, fp, fn, tp = confusion_matrix(y_true, (proba >= t).astype(int), labels=[0, 1]).ravel()
    return COST_FN * fn + COST_FP * fp

thresholds = np.linspace(0.02, 0.9, 89)
costs = np.array([total_cost(y_train, oof, t) for t in thresholds])
best_t = float(thresholds[costs.argmin()])
print(f"cost-minimising threshold on out-of-fold predictions: {best_t:.2f} (theory for calibrated probs: 40/340 = {40 / 340:.2f})")

final.fit(X_train, y_train)
p_test = final.predict_proba(X_test)[:, 1]
for label, t in [("default 0.50", 0.5), (f"tuned {best_t:.2f}", best_t)]:
    tn, fp, fn, tp = confusion_matrix(y_test, (p_test >= t).astype(int)).ravel()
    print(f"TEST @ {label:12}: flagged {tp + fp:4d}, caught {tp:3d}/{tp + fn} churners, false alarms {fp:3d} "
          f"-> cost ${total_cost(y_test, p_test, t):,}")
print(f"TEST ROC-AUC {roc_auc_score(y_test, p_test):.3f} | PR-AUC {average_precision_score(y_test, p_test):.3f}")
assert total_cost(y_test, p_test, best_t) < total_cost(y_test, p_test, 0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(thresholds, costs / 1000, color="#4c6ef5", lw=2)
ax.axvline(0.5, color="grey", ls="--", label="default 0.5")
ax.axvline(best_t, color="#e03131", ls="-", label=f"cost-optimal {best_t:.2f}")
ax.set_xlabel("decision threshold"); ax.set_ylabel("expected cost, $k (train, out-of-fold)")
ax.set_title("A missed churner costs 7.5x a wasted offer, so the best threshold is low"); ax.legend()
plt.tight_layout(); plt.show()

Why so low? The decision rule is to call a customer if `p × 300 > 40`, that is if `p > 40/340 ≈ 0.12`. The out-of-fold search lands close to that theoretical value **because logistic regression's probabilities are well calibrated**. With a poorly calibrated model (a boosted model or an SVM, say), you would calibrate first (`CalibratedClassifierCV`) or trust the empirical search. If the team can only make N calls a week, drop the threshold idea and call the top N by probability.

## Step 7 · Which features matter? Permutation importance on held-out data

Shuffle one column at a time in the **test** set and measure how much ROC-AUC drops. It is model-agnostic, it works through the whole pipeline on the raw columns, and it avoids impurity importance's bias towards high-cardinality features.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(final, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=0, n_jobs=1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.barh(imp.index, imp.values, xerr=perm.importances_std[np.argsort(perm.importances_mean)], color="#4c6ef5", capsize=3)
ax.set_xlabel("drop in test ROC-AUC when the column is shuffled"); ax.set_title("Permutation importance (held-out)")
plt.tight_layout(); plt.show()
coefs = pd.Series(final.named_steps["model"].coef_[0], index=final.named_steps["prep"].get_feature_names_out())
print("largest logistic coefficients (log-odds per unit / per category):")
print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(6).round(2).to_string())
assert imp.idxmax() in {"contract", "tenure_months"}

Contract type and tenure dominate, followed by support calls and payment method, which matches how the data was generated. So the method recovers the truth when there is a truth to recover. Say the caveat too: importance is **predictive, not causal**. "Month-to-month customers churn" does not mean forcing annual contracts will stop churn.

## Step 8 · Error analysis: where is the model wrong?

Averages hide the failures. Slice the test errors by segment at the tuned threshold.

In [ ]:
test = X_test.assign(churned=y_test.values, p=p_test, flag=(p_test >= best_t).astype(int))
test["error"] = np.select([(test.flag == 0) & (test.churned == 1), (test.flag == 1) & (test.churned == 0)],
                          ["missed churner", "false alarm"], "ok")
test["tenure_band"] = pd.cut(test.tenure_months, [0, 6, 12, 24, 72], labels=["0-6", "7-12", "13-24", "25-72"])
by_contract = pd.crosstab(test.contract, test.error, normalize="index").round(2)
print("error mix by contract (row = 100%):\n", by_contract.to_string())
missed = test[test.error == "missed churner"]
print(f"\n{len(missed)} missed churners; their median tenure {missed.tenure_months.median():.0f} months vs "
      f"{test[test.churned == 1].tenure_months.median():.0f} for all churners; contracts:",
      missed.contract.value_counts().to_dict())

The pattern to report: the misses are churners who **look like loyal customers**, with long tenure and one- or two-year contracts. Nothing in these features says they are about to leave. The next move is new *features*, not a new model. Examples: usage trend over the last 30 days, recent complaints, price changes, whether a contract end date is coming up. That is the kind of conclusion a reviewer wants from a take-home.

## Step 9 · Save, load, and a `predict()` function

Serialise the **whole fitted pipeline**, preprocessing included, so the endpoint cannot apply different preprocessing than training did. Joblib files are pickles, so load only files you produced (the same warning as with the FAISS index in [the RAG skeleton](01_rag_skeleton.ipynb)).

In [ ]:
import tempfile
from pathlib import Path

import joblib

MODEL_PATH = Path(tempfile.mkdtemp(prefix="churn_")) / "churn_pipeline_v1.joblib"
joblib.dump({"pipeline": final, "threshold": best_t, "features": NUMERIC + CATEGORICAL, "version": "v1"}, MODEL_PATH)
bundle = joblib.load(MODEL_PATH)
assert np.allclose(bundle["pipeline"].predict_proba(X_test)[:, 1], p_test)
print(f"saved {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1024:.0f} KB); reloaded model gives identical probabilities")

def predict(customer: dict) -> dict:
    row = pd.DataFrame([customer], columns=bundle["features"])
    p = float(bundle["pipeline"].predict_proba(row)[0, 1])
    return {"churn_probability": round(p, 4), "call_customer": p >= bundle["threshold"],
            "threshold": round(bundle["threshold"], 3), "model_version": bundle["version"]}

new_customer = {"tenure_months": 2, "monthly_charges": 95.0, "contract": "month-to-month",
                "payment_method": "electronic check", "internet": "fiber", "support_calls": 4, "senior": 0}
loyal_customer = {**new_customer, "tenure_months": 60, "contract": "two-year", "payment_method": "card", "support_calls": 0}
print("risky  :", predict(new_customer))
print("loyal  :", predict(loyal_customer))
print("unseen :", predict({**new_customer, "payment_method": "crypto"}))     # handle_unknown="ignore"

## Step 10 · A FastAPI endpoint, tested with `TestClient`

The pydantic model **is** the contract. It uses `Literal` categories and numeric ranges, so a request with an impossible tenure or a typo in the contract type is a 422, not a silent garbage prediction.

In [ ]:
from typing import Literal

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class Customer(BaseModel):
    tenure_months: int = Field(ge=0, le=600)
    monthly_charges: float | None = Field(default=None, ge=0, le=1000)   # None -> imputed with the training median
    contract: Literal["month-to-month", "one-year", "two-year"]
    payment_method: str = Field(min_length=1, max_length=40)             # unknown values are tolerated
    internet: Literal["dsl", "fiber", "none"]
    support_calls: int = Field(ge=0, le=100)
    senior: Literal[0, 1]

app = FastAPI(title="churn-model")

@app.post("/predict")
def predict_endpoint(customer: Customer):
    return predict(customer.model_dump())

@app.get("/healthz")
def healthz():
    return {"ok": True, "model_version": bundle["version"]}

api = TestClient(app)
r = api.post("/predict", json=new_customer)
print(r.status_code, r.json())
assert r.status_code == 200 and r.json()["call_customer"] is True
assert api.post("/predict", json={**loyal_customer}).json()["call_customer"] is False
assert api.post("/predict", json={**new_customer, "contract": "monthly"}).status_code == 422
assert api.post("/predict", json={**new_customer, "tenure_months": -3}).status_code == 422
assert api.post("/predict", json={**new_customer, "monthly_charges": None}).status_code == 200
print("endpoint tests passed:", api.get("/healthz").json())

## The follow-ups they will ask

| Ask | Answer |
|---|---|
| Class imbalance? | About 28% of customers churn here, which is mild. Use stratified splits, PR-AUC, and a cost-based threshold. `class_weight="balanced"` or resampling changes the probabilities, so re-tune the threshold after using it. |
| Why not accuracy? | Always predicting "stays" already scores about 72%. Accuracy ignores the unequal costs. |
| How do you know it generalises? | 5-fold CV for model choice, and a test set touched **once** at the end. For time-stamped data, a time-based split, or backtesting on the last months. |
| Is the model calibrated? | Check a reliability curve and the Brier score. Logistic regression usually is. Calibrate boosted models before using the probabilities as costs. |
| Monitoring? | Input drift (PSI or KS per feature), prediction drift, the rate of unknown categories, and, once labels arrive, realised precision and recall and the cost. Retrain on a schedule or when drift fires. |
| Fairness? | Check error rates by sensitive groups (here `senior`). Unequal false-alarm or miss rates are a product decision, not only a metric. |

## The whole thing as one file: `train.py` + `serve.py`

```python
"""train.py - churn model: leakage-safe pipeline, CV model choice, cost-based threshold, saved bundle.
pip install pandas scikit-learn joblib ; usage: python train.py customers.csv"""
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["tenure_months", "monthly_charges", "support_calls", "senior"]
CATEGORICAL = ["contract", "payment_method", "internet"]
COST_FN, COST_FP = 300, 40                              # missed churner vs wasted retention offer

df = pd.read_csv(sys.argv[1])
X, y = df[NUMERIC + CATEGORICAL], df["churned"]         # only columns known BEFORE the outcome
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])),
    ("model", LogisticRegression(max_iter=1000)),
])

def cost(y_true, proba, t):
    tn, fp, fn, tp = confusion_matrix(y_true, (proba >= t).astype(int), labels=[0, 1]).ravel()
    return COST_FN * fn + COST_FP * fp

oof = cross_val_predict(pipe, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                        method="predict_proba")[:, 1]
thresholds = np.linspace(0.02, 0.9, 89)
best_t = float(thresholds[np.argmin([cost(y_train, oof, t) for t in thresholds])])

pipe.fit(X_train, y_train)
p_test = pipe.predict_proba(X_test)[:, 1]
print(f"test ROC-AUC {roc_auc_score(y_test, p_test):.3f}; threshold {best_t:.2f}; "
      f"test cost ${cost(y_test, p_test, best_t):,} vs ${cost(y_test, p_test, 0.5):,} at 0.5")
joblib.dump({"pipeline": pipe, "threshold": best_t, "features": NUMERIC + CATEGORICAL, "version": "v1"},
            "churn_pipeline_v1.joblib")
```

```python
"""serve.py - pip install fastapi uvicorn pandas scikit-learn joblib ; run: uvicorn serve:app"""
from typing import Literal

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

bundle = joblib.load("churn_pipeline_v1.joblib")        # a pickle: only load files you produced

class Customer(BaseModel):
    tenure_months: int = Field(ge=0, le=600)
    monthly_charges: float | None = Field(default=None, ge=0, le=1000)
    contract: Literal["month-to-month", "one-year", "two-year"]
    payment_method: str = Field(min_length=1, max_length=40)
    internet: Literal["dsl", "fiber", "none"]
    support_calls: int = Field(ge=0, le=100)
    senior: Literal[0, 1]

app = FastAPI(title="churn-model")

@app.post("/predict")
def predict(customer: Customer):
    row = pd.DataFrame([customer.model_dump()], columns=bundle["features"])
    p = float(bundle["pipeline"].predict_proba(row)[0, 1])
    return {"churn_probability": round(p, 4), "call_customer": p >= bundle["threshold"],
            "model_version": bundle["version"]}
```

## Try it yourself

**1.** The retention team can only make **100 calls** per test period. Drop the threshold and call the 100 highest-probability customers. What is the precision@100, how many churners do you catch, and what is the cost?

In [ ]:
# Solution — try it yourself first, then run this
top100 = np.argsort(-p_test)[:100]
caught = int(y_test.to_numpy()[top100].sum())
missed = int(y_test.sum()) - caught
print(f"precision@100 = {caught / 100:.2f}; caught {caught}/{int(y_test.sum())} churners; "
      f"cost = ${COST_FN * missed + COST_FP * (100 - caught):,} (vs ${total_cost(y_test, p_test, best_t):,} with unlimited calls)")

**2.** Check calibration: bin the test probabilities into 5 quantile bins and compare the mean predicted probability with the actual churn rate in each bin. Do it for logistic regression and for the random forest.

In [ ]:
# Solution — try it yourself first, then run this
from sklearn.calibration import calibration_curve

rf = Pipeline([("prep", preprocess), ("model", MODELS["random forest"])]).fit(X_train, y_train)
for name, proba in [("logistic", p_test), ("random forest", rf.predict_proba(X_test)[:, 1])]:
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=5, strategy="quantile")
    print(f"{name:14} predicted {np.round(mean_pred, 2)}  actual {np.round(frac_pos, 2)}  "
          f"max gap {np.abs(frac_pos - mean_pred).max():.2f}")

**3.** Prove the leak matters end to end: train the same pipeline *with* `cancellation_fee` added as a numeric feature and compare test ROC-AUC. Then explain in one sentence why this model would fail in production.

In [ ]:
# Solution — try it yourself first, then run this
Xl_train, Xl_test = X_all.loc[X_train.index], X_all.loc[X_test.index]
leaky = Pipeline([("prep", ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC + ["cancellation_fee"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])), ("model", LogisticRegression(max_iter=1000))])
leaky.fit(Xl_train, y_train)
print(f"test ROC-AUC with the leak: {roc_auc_score(y_test, leaky.predict_proba(Xl_test)[:, 1]):.3f}  "
      f"vs honest model: {roc_auc_score(y_test, p_test):.3f}")
# In production the fee is 0 for everyone we are trying to score (they haven't cancelled yet),
# so the model's favourite feature carries no information exactly when it is needed.

## Say it in an interview

**30 seconds:** "First, EDA for target rate, drivers and suspicious columns. I found and dropped a post-outcome column that gave AUC 0.9+ on its own. Then a stratified split, preprocessing inside a `ColumnTransformer` + `Pipeline` so CV is leak-free, a dummy baseline, and three models under 5-fold CV. Logistic regression tied the ensembles, so I kept the simple, calibrated one. I set the threshold from the business costs on out-of-fold predictions, which came out near the theoretical 40/340, and reported the test set once. Permutation importance says contract and tenure dominate. The misses are long-tenure churners, which points to new features rather than a new model. The whole pipeline is saved with joblib and served behind a validated FastAPI endpoint."

**Numbers:** threshold ≈ C_FP / (C_FP + C_FN) for calibrated probabilities. With about 28% positives, accuracy is useless (72% for free), and PR-AUC is more informative than ROC-AUC. Use 5-fold stratified CV and touch the test set once.

**Traps:** fitting the scaler or imputer before splitting; tuning the threshold on the test set; post-outcome features; `fit_transform` on test data; not handling unseen categories; saving only the model without its preprocessing; reading impurity importance as causal; reporting 0.5-threshold metrics as if 0.5 meant something.

## Next

- Foundations: [ML workflow and splits](../05_machine_learning/01_ml_workflow_and_train_test_split.ipynb) · [logistic regression](../05_machine_learning/03_logistic_regression_classification.ipynb) · [metrics, ROC, PR](../05_machine_learning/04_metrics_confusion_matrix_roc_pr.ipynb) · [overfitting and cross-validation](../05_machine_learning/05_overfitting_regularization_cross_validation.ipynb) · [trees, forests, boosting](../05_machine_learning/06_decision_trees_random_forest_boosting.ipynb) · [feature engineering and pipelines](../05_machine_learning/09_feature_engineering_and_pipelines.ipynb)
- Other rounds: [chat API with FastAPI](03_chat_api_with_fastapi.ipynb) · [implement-from-scratch drills](07_implement_from_scratch_drills.ipynb)
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html)